# Détection d'intrusions — Log System
## Projet IDS | Apprentissage non supervisé — LOF

In [1]:
import pandas as pd
import os

# Se placer à la racine du projet IDS
os.chdir(r'C:\Users\asus\Desktop\IDS')

# Charger le fichier
df = pd.read_csv('Non supervisé/Data/System_ohp-win12.csv')
print(f"✅ Shape: {df.shape}")
print(df.head())

✅ Shape: (14613, 6)
         Level          Date and Time                   Source  Event ID  \
0        Error  12/19/2016 3:46:14 PM                 Schannel   36888.0   
1        Error  12/19/2016 3:46:14 PM                 Schannel   36888.0   
2  Information  12/19/2016 3:39:38 PM  Service Control Manager    7036.0   
3  Information  12/19/2016 3:39:17 PM  Service Control Manager    7036.0   
4  Information  12/19/2016 3:38:47 PM  Service Control Manager    7036.0   

  Task Category                                            Message  
0           NaN  A fatal alert was generated and sent to the re...  
1           NaN  A fatal alert was generated and sent to the re...  
2           NaN  The Application Experience service entered the...  
3           NaN  The Software Protection service entered the st...  
4           NaN  The Software Protection service entered the ru...  


## 2. Nettoyage et feature engineering
Extraction features temporelles et textuelles depuis les messages système Windows.

In [2]:
df_clean = df.copy()

# 1. Nettoyer les noms de colonnes
df_clean.columns = df_clean.columns.str.strip()

# 2. Convertir la date
df_clean['DateTime'] = pd.to_datetime(df_clean['Date and Time'], errors='coerce')

# 3. Extraire des features temporelles
df_clean['hour'] = df_clean['DateTime'].dt.hour
df_clean['day_of_week'] = df_clean['DateTime'].dt.dayofweek
df_clean['minute'] = df_clean['DateTime'].dt.minute

# 4. Remplacer les valeurs manquantes
df_clean['Task Category'] = df_clean['Task Category'].fillna('None')
df_clean['Message'] = df_clean['Message'].fillna('')

# 5. Features textuelles à partir du Message
df_clean['msg_length'] = df_clean['Message'].str.len()
df_clean['has_fatal'] = df_clean['Message'].str.contains('fatal', case=False).astype(int)
df_clean['has_error'] = df_clean['Message'].str.contains('error', case=False).astype(int)
df_clean['has_tls'] = df_clean['Message'].str.contains('tls', case=False).astype(int)
df_clean['has_alert'] = df_clean['Message'].str.contains('alert', case=False).astype(int)

# 6. Encoder le niveau (Level)
df_clean['is_error_level'] = (df_clean['Level'] == 'Error').astype(int)
df_clean['is_warning_level'] = (df_clean['Level'] == 'Warning').astype(int)
df_clean['is_info_level'] = (df_clean['Level'] == 'Information').astype(int)

# 7. Compter les mots dans le message
df_clean['word_count'] = df_clean['Message'].str.split().str.len()

print(f"\n✅ Shape après nettoyage: {df_clean.shape}")
print(f"✅ Lignes avec erreur: {df_clean['is_error_level'].sum()}")
print(f"\nColonnes créées: {df_clean.columns.tolist()}")

C:\Users\asus\AppData\Local\Temp\ipykernel_3296\580598680.py:7: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df_clean['DateTime'] = pd.to_datetime(df_clean['Date and Time'], errors='coerce')



✅ Shape après nettoyage: (14613, 19)
✅ Lignes avec erreur: 297

Colonnes créées: ['Level', 'Date and Time', 'Source', 'Event ID', 'Task Category', 'Message', 'DateTime', 'hour', 'day_of_week', 'minute', 'msg_length', 'has_fatal', 'has_error', 'has_tls', 'has_alert', 'is_error_level', 'is_warning_level', 'is_info_level', 'word_count']


## 3. Preprocessing — encodage et normalisation

In [3]:
from sklearn.preprocessing import LabelEncoder, StandardScaler, MinMaxScaler
from sklearn.compose import ColumnTransformer

print("\n" + "="*50)
print("ÉTAPE 2: PREPROCESSING")
print("="*50)

# ========== COLONNES À UTILISER ==========
# Colonnes numériques
numerical_cols = ['Event ID', 'hour', 'day_of_week', 'minute', 
                  'msg_length', 'word_count', 'has_fatal', 'has_error', 
                  'has_tls', 'has_alert', 'is_error_level', 
                  'is_warning_level', 'is_info_level']

# Colonnes catégorielles à encoder
categorical_cols = ['Source', 'Task Category']

# ========== ENCODAGE ==========
df_encoded = df_clean.copy()
encoders = {}

for col in categorical_cols:
    if col in df_encoded.columns:
        le = LabelEncoder()
        df_encoded[col + '_encoded'] = le.fit_transform(df_encoded[col].astype(str))
        encoders[col] = le
        print(f"✓ {col} encodé ({(df_encoded[col + '_encoded'].nunique())} valeurs uniques)")
        df_encoded = df_encoded.drop(columns=[col])

# ========== SÉLECTION DES FEATURES ==========
feature_cols = numerical_cols + [c for c in df_encoded.columns if '_encoded' in c]

# S'assurer que toutes les colonnes existent
feature_cols = [col for col in feature_cols if col in df_encoded.columns]

X = df_encoded[feature_cols].values

print(f"\n✅ Features sélectionnées ({len(feature_cols)}):")
for col in feature_cols:
    print(f"  - {col}")

# ========== SCALING ==========
# Pour LOF, StandardScaler est recommandé
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# Alternative: MinMaxScaler (si préféré)
# scaler = MinMaxScaler()
# X_scaled = scaler.fit_transform(X)

print(f"\n✅ Shape finale: {X_scaled.shape}")
print(f"✅ Stats scaling - min: {X_scaled.min():.3f}, max: {X_scaled.max():.3f}")


ÉTAPE 2: PREPROCESSING
✓ Source encodé (30 valeurs uniques)
✓ Task Category encodé (16 valeurs uniques)

✅ Features sélectionnées (15):
  - Event ID
  - hour
  - day_of_week
  - minute
  - msg_length
  - word_count
  - has_fatal
  - has_error
  - has_tls
  - has_alert
  - is_error_level
  - is_warning_level
  - is_info_level
  - Source_encoded
  - Task Category_encoded

✅ Shape finale: (14613, 15)
✅ Stats scaling - min: nan, max: nan


## 4. Modèle — Local Outlier Factor (LOF)
LOF compare la densité locale de chaque point avec celle de ses voisins.
Un point dans une zone peu dense est considéré comme une anomalie.

In [4]:
from sklearn.neighbors import LocalOutlierFactor
from sklearn.impute import SimpleImputer
import numpy as np

print("=== MODÈLE LOF ===")

# Imputation des NaN restants
imputer = SimpleImputer(strategy='median')
X_imputed = imputer.fit_transform(X_scaled)
print(f"Shape après imputation : {X_imputed.shape}")

# Modèle LOF
lof = LocalOutlierFactor(
    n_neighbors  = 50,
    contamination= 0.2,
    novelty      = False,
    metric       = 'euclidean'
)

predictions    = lof.fit_predict(X_imputed)
anomaly_scores = lof.negative_outlier_factor_
anomalies      = predictions == -1

print(f"\n✅ Résultats LOF :")
print(f"   - Normal    : {(predictions == 1).sum()}")
print(f"   - Anomalies : {(predictions == -1).sum()}")
print(f"   - Taux      : {100*(predictions==-1).sum()/len(predictions):.1f}%")

=== MODÈLE LOF ===
Shape après imputation : (14613, 15)

✅ Résultats LOF :
   - Normal    : 11690
   - Anomalies : 2923
   - Taux      : 20.0%


c:\Users\asus\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\neighbors\_lof.py:325: UserWarning: Duplicate values are leading to incorrect results. Increase the number of neighbors for more accurate results.
  warnings.warn(


## 5. Export et sauvegarde

In [5]:
import pandas as pd, os

base = r"C:\Users\asus\Desktop\IDS\Non supervisé\Notebooks\outputs"
os.makedirs(base, exist_ok=True)

df_system_normalized = pd.DataFrame({
    "timestamp"     : df_clean["Date and Time"].values
                      if "Date and Time" in df_clean.columns else "unknown",
    "source_ip"     : "unknown",
    "dest_ip"       : "unknown",
    "log_source"    : "system",
    "source_name"   : df_clean["Source"].values
                      if "Source" in df_clean.columns else "unknown",
    "event_id"      : df_clean["Event ID"].values
                      if "Event ID" in df_clean.columns else "unknown",
    "level"         : df_clean["Level"].values
                      if "Level" in df_clean.columns else "unknown",
    "anomaly_score" : anomaly_scores,
    "prediction"    : predictions,
    "alert_level"   : ["HIGH" if p == -1 else "NORMAL" for p in predictions],
})

df_system_normalized.to_csv(
    os.path.join(base, "system_output_normalized.csv"), index=False
)
print("✅ system_output_normalized.csv sauvegardé")
print(f"   {len(df_system_normalized)} lignes")
print(df_system_normalized.head())

✅ system_output_normalized.csv sauvegardé
   14613 lignes
               timestamp source_ip  dest_ip log_source  \
0  12/19/2016 3:46:14 PM   unknown  unknown     system   
1  12/19/2016 3:46:14 PM   unknown  unknown     system   
2  12/19/2016 3:39:38 PM   unknown  unknown     system   
3  12/19/2016 3:39:17 PM   unknown  unknown     system   
4  12/19/2016 3:38:47 PM   unknown  unknown     system   

               source_name  event_id        level  anomaly_score  prediction  \
0                 Schannel   36888.0        Error      -1.047076           1   
1                 Schannel   36888.0        Error      -1.047076           1   
2  Service Control Manager    7036.0  Information      -0.997688           1   
3  Service Control Manager    7036.0  Information      -0.997353           1   
4  Service Control Manager    7036.0  Information      -0.991971           1   

  alert_level  
0      NORMAL  
1      NORMAL  
2      NORMAL  
3      NORMAL  
4      NORMAL  


In [6]:
import joblib, os
base = r"C:\Users\asus\Desktop\IDS\models"
os.makedirs(base, exist_ok=True)
joblib.dump(lof,    base + r"\system_lof.pkl")
joblib.dump(scaler, base + r"\system_scaler.pkl")
print("✅ system model sauvegardé")

✅ system model sauvegardé
